# Gemma 4 E4B (tyv LoRA): сокращение словаря и удаление зрения/звука

Цель — модель, которая влезет в iPhone. У Gemma 4 E4B словарь 262 144 токена, и он хранится
в двух таблицах: обычные эмбеддинги (262144 × 2560 ≈ 0.67 млрд параметров) и **эмбеддинги по слоям, PLE**
(262144 × 42 слоя × 256 ≈ 2.8 млрд). Это ~44% всех весов. Оставляем только токены русского и тувинского.

Что делает ноутбук (всё автоматически, «Выполнить все»):
1. Собирает данные тем же `prepare_data.py` (с GitHub) — тот же сплит и чистый тест.
2. Находит все токены, которые встречаются в **обучающих** данных (тест не подсматриваем), + служебные,
   байтовые и одиночные символы. Для BPE добавляет «предков» каждого токена, чтобы токенизация не менялась.
3. Собирает **новый токенизатор** с короткими номерами и проверяет, что он режет текст точно так же.
4. Сливает адаптер, оставляет **только языковую модель** (без зрения и звука), вырезает строки таблиц словаря.
5. Оценивает на чистом тесте и сравнивает с полной моделью (50.5 / 50.9), сохраняет на Google Диск.
6. По желанию — MLX 8 и 4 бита и загрузка на Hugging Face (приватно).

**Среда:** A100 + High-RAM (нужно ~60 ГБ ОЗУ для слияния). Займёт ~1–1.5 часа.
Каждая ячейка проверяет результат и останавливается с понятной ошибкой, если что-то не так.

In [ ]:
# Ячейка 1 — библиотеки
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip uninstall -y -q torchao
!pip install -q -U transformers peft accelerate datasets sacrebleu
import torch, transformers, peft
print("torch", torch.__version__, "| transformers", transformers.__version__, "| peft", peft.__version__)

In [ ]:
# Ячейка 2 — Google Диск и (необязательно) Hugging Face
from google.colab import drive
drive.mount('/content/drive')
from huggingface_hub import login
try:
    from google.colab import userdata
    login(token=userdata.get("HF_TOKEN")); print("Вошёл в Hugging Face")
except Exception:
    print("HF_TOKEN не задан — загрузка на Hugging Face будет недоступна")

In [ ]:
# Ячейка 3 — настройки
import os
MODEL_ID   = "google/gemma-4-E4B-it"
ADAPTER    = "Agisight/tyv-gemma4-e4b-lora"
GITHUB_RAW = "https://raw.githubusercontent.com/Agisight/tyv-translation-mlx/main"
LOCAL      = "/content/pruned"                                      # модель в bf16 (рабочая копия)
OUT_DIR    = "/content/drive/MyDrive/tyv-llm/gemma4-e4b-tyv-pruned" # копия на Диске
HUB_REPO   = "Agisight/tyv-gemma4-e4b-pruned"
PUSH_BF16  = False   # True — загрузить сокращённую bf16-модель на HF (приватно); нужна на Маке для своих вариантов квантизации
RUN_MLX    = True    # конвертация в MLX 8/4 бита (на CPU, ~10–20 минут)
PUSH_MLX   = False   # True — загрузить MLX-версии на HF (приватно); без этого они только на Диске, и `hf download` их не найдёт
os.makedirs(LOCAL, exist_ok=True); os.makedirs(OUT_DIR, exist_ok=True)

In [ ]:
# Ячейка 4 — данные: тот же prepare_data.py, что в репозитории (версия датасета зафиксирована)
import urllib.request
if not os.path.exists("prepare_data.py"):
    urllib.request.urlretrieve(f"{GITHUB_RAW}/prepare_data.py", "prepare_data.py")
if not os.path.exists("data/train.jsonl"):
    !python prepare_data.py --train-limit 0 --out data
!wc -l data/*.jsonl
# переводы полной модели — для сравнения
urllib.request.urlretrieve(f"{GITHUB_RAW}/eval_results_gemma4.json", "eval_results_gemma4.json")

In [ ]:
# Ячейка 5 — какие токены оставить (только по обучающим данным)
import json, re
from transformers import AutoTokenizer
tok = AutoTokenizer.from_pretrained(ADAPTER)
V0 = 262144
train = [json.loads(l) for l in open("data/train.jsonl", encoding="utf-8")]
print("обучающих примеров:", len(train))

texts = [tok.apply_chat_template(ex["messages"], tokenize=False) for ex in train]
texts += [tok.apply_chat_template(ex["messages"][:1], tokenize=False, add_generation_prompt=True) for ex in train[:2000]]
print("пример:", repr(texts[0][:200]))
used = set()
for i in range(0, len(texts), 20000):
    for ids in tok(texts[i:i + 20000], add_special_tokens=False)["input_ids"]:
        used.update(ids)
print("токенов в обучающих данных:", len(used))

vocab = tok.get_vocab()
extra = set(tok.all_special_ids)
extra |= {i for t, i in vocab.items() if re.fullmatch(r"<0x[0-9A-Fa-f]{2}>", t)}          # байтовый запасной вариант
chars = [chr(c) for c in range(32, 127)] + list("АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯабвгдеёжзийклмнопрстуфхцчшщъыьэюя") \
        + list("ҢңӨөҮүҺһ«»—–…№„“”’‘•·€₽°×")
for ch in chars:
    for t in (ch, "▁" + ch):
        if t in vocab: extra.add(vocab[t])
kept = used | extra
print(f"служебных/байтовых/символов добавлено: {len(extra - used)}")

# BPE: добавляем «предков» — промежуточные токены, через которые BPE приходит к оставленным
js = json.loads(tok.backend_tokenizer.to_str())
MODEL_TYPE = js["model"]["type"]
print("тип токенизатора:", MODEL_TYPE, "| byte_fallback:", js["model"].get("byte_fallback"))
id2tok = {i: t for t, i in js["model"]["vocab"].items()}
if MODEL_TYPE == "BPE":
    # Один токен может получаться РАЗНЫМИ слияниями («Wi-» + «Fi» или «Wi» + «-Fi»), поэтому
    # берём части ВСЕХ слияний, дающих оставленный токен, — иначе путь BPE может оборваться.
    parents = {}
    for m in js["model"]["merges"]:
        a, b = (m.split(" ", 1) if isinstance(m, str) else m)
        parents.setdefault(a + b, []).append((a, b))
    stack = [id2tok[i] for i in kept if i in id2tok]
    seen = set(stack)
    while stack:
        t = stack.pop()
        for pair in parents.get(t, ()):
            for p in pair:
                if p not in seen and p in js["model"]["vocab"]:
                    seen.add(p); stack.append(p)
    before = len(kept)
    kept |= {js["model"]["vocab"][t] for t in seen}
    print(f"предков BPE добавлено: {len(kept) - before}")
KEPT = sorted(i for i in kept if i < V0)
old2new = {o: n for n, o in enumerate(KEPT)}
N = len(KEPT)
assert all(old2new.get(i) == i for i in (0, 1, 2)), "pad/eos/bos должны остаться 0/1/2"
print(f"ИТОГО словарь: {N:,} токенов вместо {V0:,} ({N / V0:.1%})")

# сколько токенов теста не попало (тест не использовался при выборе)
pairs = [json.loads(l) for l in open("data/eval_pairs.jsonl", encoding="utf-8")]
tt = tok([p["src"] for p in pairs] + [p["ref"] for p in pairs], add_special_tokens=False)["input_ids"]
tot = sum(len(x) for x in tt); oov = sum(i not in old2new for x in tt for i in x)
print(f"токенов теста вне словаря: {oov} из {tot} ({oov / tot:.3%})")
json.dump({"kept_old_ids": KEPT, "old_vocab_size": V0}, open(f"{LOCAL}/vocab_prune_map.json", "w"))

In [ ]:
# Ячейка 6 — новый токенизатор с короткими номерами + проверка, что режет текст так же
import copy, shutil, tempfile
assert MODEL_TYPE == "BPE", f"Токенизатор {MODEL_TYPE} — нужна другая схема сокращения, пришли вывод ячейки 5"
nj = copy.deepcopy(js)
nv = {id2tok[o]: old2new[o] for o in KEPT if o in id2tok}
nj["model"]["vocab"] = nv
nm = []
for m in js["model"]["merges"]:
    a, b = (m.split(" ", 1) if isinstance(m, str) else m)
    if a in nv and b in nv and (a + b) in nv:
        nm.append(m)
nj["model"]["merges"] = nm
nj["added_tokens"] = [dict(t, id=old2new[t["id"]]) for t in js.get("added_tokens", []) if t["id"] in old2new]

def remap_ids(obj):   # пост-процессор и пр. могут хранить номера служебных токенов
    if isinstance(obj, dict):
        for k, v in obj.items():
            if k in ("id", "ids") and isinstance(v, list) and all(isinstance(x, int) for x in v):
                obj[k] = [old2new.get(x, x) for x in v]
            else:
                remap_ids(v)
    elif isinstance(obj, list):
        for x in obj: remap_ids(x)
remap_ids(nj.get("post_processor"))
print(f"merges: {len(js['model']['merges']):,} → {len(nm):,}")

tmp = tempfile.mkdtemp()
tok.save_pretrained(tmp)
for f in os.listdir(tmp):
    if f != "tokenizer.json": shutil.copy(f"{tmp}/{f}", f"{LOCAL}/{f}")
json.dump(nj, open(f"{LOCAL}/tokenizer.json", "w", encoding="utf-8"), ensure_ascii=False)
tc_path = f"{LOCAL}/tokenizer_config.json"
tc = json.load(open(tc_path, encoding="utf-8"))
if "added_tokens_decoder" in tc:
    tc["added_tokens_decoder"] = {str(old2new[int(k)]): v for k, v in tc["added_tokens_decoder"].items() if int(k) in old2new}
json.dump(tc, open(tc_path, "w", encoding="utf-8"), ensure_ascii=False, indent=2)

ntok = AutoTokenizer.from_pretrained(LOCAL)
check = [p["src"] for p in pairs] + [p["ref"] for p in pairs] + texts[:5000]
bad = 0; examples = []
for t in check:
    a = tok(t, add_special_tokens=False)["input_ids"]; b = ntok(t, add_special_tokens=False)["input_ids"]
    if all(i in old2new for i in a) and [old2new[i] for i in a] != b:
        bad += 1
        if len(examples) < 3:
            j = next(k for k, (x, y) in enumerate(zip([old2new[i] for i in a], b)) if x != y) if b else 0
            examples.append(f"{t[:80]!r}\n      было: {tok.convert_ids_to_tokens(a[j:j+4])}\n      стало: {ntok.convert_ids_to_tokens(b[j:j+4])}")
print(f"проверено {len(check):,} текстов; токенизация отличается: {bad}")
for e in examples: print("   ", e)
p_old = tok.apply_chat_template([{"role": "user", "content": "ru→tyv: Цель: Луна."}], add_generation_prompt=True, tokenize=False)
p_new = ntok.apply_chat_template([{"role": "user", "content": "ru→tyv: Цель: Луна."}], add_generation_prompt=True, tokenize=False)
assert p_old == p_new, "шаблон чата изменился"
assert bad == 0, "новый токенизатор режет текст иначе — пришли вывод"
print("Токенизатор ОК:", len(ntok), "токенов")

In [ ]:
# Ячейка 7 — слить адаптер, оставить только языковую модель, вырезать строки словаря
import importlib, gc
from transformers import AutoModelForCausalLM
from peft import PeftModel
full = AutoModelForCausalLM.from_pretrained(MODEL_ID, dtype=torch.bfloat16, device_map="cpu")
full = PeftModel.from_pretrained(full, ADAPTER).merge_and_unload()
print("класс полной модели:", type(full).__name__)

lm_name, lm = next((n, m) for n, m in full.named_modules() if type(m).__name__.endswith("TextModel"))
print("языковая модель:", lm_name, type(lm).__name__)
text_cfg = getattr(full.config, "text_config", full.config)
mod = importlib.import_module(type(full).__module__)
CausalLM = next(getattr(mod, n) for n in dir(mod) if n.endswith("ForCausalLM") and "Conditional" not in n)
print("текстовый класс:", CausalLM.__name__)

torch.set_default_dtype(torch.bfloat16)
text = CausalLM(text_cfg)
torch.set_default_dtype(torch.float32)
missing, unexpected = text.model.load_state_dict(lm.state_dict(), strict=False)
print("missing:", missing[:5], "| unexpected:", unexpected[:5])
assert not missing and not unexpected, "веса языковой модели не совпали по именам — пришли вывод"
if getattr(text.config, "tie_word_embeddings", True):
    text.tie_weights()
else:
    text.lm_head.weight.data.copy_(full.lm_head.weight.data)
gen_cfg = full.generation_config
del full, lm; gc.collect()

idx = torch.tensor(KEPT)
sliced = []
for name, m in text.named_modules():
    if isinstance(m, torch.nn.Embedding) and m.num_embeddings == V0:
        m.weight = torch.nn.Parameter(m.weight.data[idx].clone())
        m.num_embeddings = N
        if m.padding_idx is not None: m.padding_idx = old2new.get(m.padding_idx, 0)
        sliced.append((name, tuple(m.weight.shape)))
print("сокращены таблицы:", sliced)
assert sliced, "не нашёл таблиц размером со словарь"
if getattr(text.config, "tie_word_embeddings", True):
    text.lm_head.weight = text.model.embed_tokens.weight
else:
    text.lm_head.weight = torch.nn.Parameter(text.lm_head.weight.data[idx].clone())
text.lm_head.out_features = N

text.config.vocab_size = N
if hasattr(text.config, "vocab_size_per_layer_input"): text.config.vocab_size_per_layer_input = N
text.config.architectures = [CausalLM.__name__]
def rmp(v): return [old2new[x] for x in v if x in old2new] if isinstance(v, list) else old2new.get(v, v)
for k in ("bos_token_id", "eos_token_id", "pad_token_id"):
    if getattr(text.config, k, None) is not None: setattr(text.config, k, rmp(getattr(text.config, k)))
    if getattr(gen_cfg, k, None) is not None: setattr(gen_cfg, k, rmp(getattr(gen_cfg, k)))
text.generation_config = gen_cfg
print("eos в generation_config:", gen_cfg.eos_token_id)

params = sum(p.numel() for p in text.parameters())
print(f"параметров: {params / 1e9:.2f} млрд (было ~8 млрд вместе со зрением и звуком)")
text.save_pretrained(LOCAL, safe_serialization=True)
!du -sh {LOCAL}

In [ ]:
# Ячейка 8 — проверка и оценка на чистом тесте (те же настройки, что у полной модели)
import re
from sacrebleu.metrics import CHRF, BLEU
from prepare_data import PROMPTS
del text; gc.collect()
model = AutoModelForCausalLM.from_pretrained(LOCAL, dtype=torch.bfloat16, device_map="cuda").eval()
ntok = AutoTokenizer.from_pretrained(LOCAL); ntok.padding_side = "left"
ref_full = {(r["direction"], r["src"]): r["hyp"] for r in json.load(open("eval_results_gemma4.json", encoding="utf-8"))}
chrf, bleu = CHRF(word_order=2), BLEU()
results = []
for d in ("ru-tyv", "tyv-ru"):
    items = [r for r in pairs if r["direction"] == d]
    hyps = []
    for i in range(0, len(items), 64):
        batch = items[i:i + 64]
        prompts = [ntok.apply_chat_template([{"role": "user", "content": PROMPTS[d].format(src=r["src"])}],
                                            add_generation_prompt=True, tokenize=False) for r in batch]
        enc = ntok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to("cuda")
        with torch.no_grad():
            out = model.generate(**enc, max_new_tokens=min(256, 32 + 3 * enc.input_ids.shape[1]), do_sample=False)
        for o in out[:, enc.input_ids.shape[1]:]:
            hyps.append(re.sub(r"<think>.*?</think>", "", ntok.decode(o, skip_special_tokens=True), flags=re.S).strip())
        if i == 0:
            for r, h in list(zip(batch, hyps))[:3]: print("   src:", r["src"], "\n   hyp:", h)
        print(f"  {d}: {min(i + 64, len(items))}/{len(items)}")
    refs = [r["ref"] for r in items]
    same = sum(h == ref_full.get((d, r["src"])) for r, h in zip(items, hyps))
    results += [{**r, "hyp": h} for r, h in zip(items, hyps)]
    print(f"{d}: chrF++ = {chrf.corpus_score(hyps, [refs]).score:.1f}  BLEU = {bleu.corpus_score(hyps, [refs]).score:.1f}"
          f"  | совпало с полной моделью: {same}/{len(items)}")
json.dump(results, open(f"{OUT_DIR}/eval_results_gemma4_pruned.json", "w"), ensure_ascii=False)
print("Полная модель (bf16): 50.5 / 50.9 | NLLB v3: 49.2 / 49.1")

In [ ]:
# Ячейка 9 — сохранить на Диск и (по желанию) на Hugging Face
!rsync -a --info=progress2 {LOCAL}/ {OUT_DIR}/bf16/
!du -sh {OUT_DIR}/bf16
if PUSH_BF16:
    from huggingface_hub import HfApi
    api = HfApi(); api.create_repo(HUB_REPO, private=True, exist_ok=True)
    api.upload_folder(repo_id=HUB_REPO, folder_path=LOCAL, commit_message="Gemma 4 E4B tyv: pruned vocabulary, text-only, bf16")
    print("Загружено:", f"https://huggingface.co/{HUB_REPO}")

In [ ]:
# Ячейка 10 — MLX 8 и 4 бита (mlx-lm, текстовая модель). Проверка перевода — на Маке.
if RUN_MLX:
    del model; gc.collect(); torch.cuda.empty_cache()
    !pip install -q -U "mlx[cpu]" mlx-lm
    for bits in (8, 4):
        out = f"/content/pruned-mlx-{bits}bit"
        if not os.path.exists(f"{out}/.converted"):
            !rm -rf {out}
            !mlx_lm.convert --hf-path {LOCAL} --mlx-path {out} -q --q-bits {bits}
            if os.path.exists(f"{out}/config.json"):
                open(f"{out}/.converted", "w").close()
        !du -sh {out}
        !rsync -a {out}/ {OUT_DIR}/mlx-{bits}bit/
        if PUSH_MLX and os.path.exists(f"{out}/.converted"):
            from huggingface_hub import HfApi
            api = HfApi(); repo = f"{HUB_REPO}-mlx-{bits}bit"
            api.create_repo(repo, private=True, exist_ok=True)
            api.upload_folder(repo_id=repo, folder_path=out, ignore_patterns=[".converted"],
                              commit_message=f"Gemma 4 E4B tyv: pruned vocabulary, text-only, MLX {bits}-bit")
            print("Загружено:", f"https://huggingface.co/{repo}")
print("Готово. Итоги: размер bf16, chrF++ из ячейки 8, размеры MLX выше. Папка на Диске:", OUT_DIR)